# OCR Quality Check

#### 1. Imports

In [1]:
import pandas as pd
from lxml import etree
import re
from jiwer import cer  # pip install jiwer
import os

#### 2. I am using the same Parsing function as in 02_xml_parsing.ipynb

In [ ]:
# namespace = {"pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}

# def parse_one_file(filepath, namespace):
#     rows = []
#     tree = etree.parse(filepath)
#     root = tree.getroot()
#     regions = root.findall(".//pc:TextRegion", namespace)
#     for region in regions:
#         custom = region.get("custom", "")
#         type_match = re.search(r"type:([^;]+);", custom)
#         region_type = type_match.group(1) if type_match else "unknown"
#         unicode_element = region.find(".//pc:Unicode", namespace)
#         if unicode_element is not None and unicode_element.text is not None:
#             text = unicode_element.text.strip()
#             rows.append({
#                 "source_file": filepath,
#                 "region_type": region_type,
#                 "text":        text
#             })
#     return rows

#### 3. Parsing the Ground Truth Files (Exported from Transkribus)

In [8]:
gt_rows = parse_one_file(
    "/Users/sophiehamann/master-thesis-code/data/processed/training_data/ground_truth/combined_xml/combined.xml",
    namespace
)

df_gt = pd.DataFrame(gt_rows)
print(df_gt.shape)
print(df_gt.head())

(1490, 3)
                                         source_file     region_type  \
0  /Users/sophiehamann/master-thesis-code/data/pr...     page_number   
1  /Users/sophiehamann/master-thesis-code/data/pr...         heading   
2  /Users/sophiehamann/master-thesis-code/data/pr...  author_creator   
3  /Users/sophiehamann/master-thesis-code/data/pr...            poem   
4  /Users/sophiehamann/master-thesis-code/data/pr...       paragraph   

                                                text  
0                                                 30  
1  The Art of Not Bowing: Writing by Women in Prison  
2                                        Carol Muske  
3                           Who the hell am I anyway  
4    In July 1973 I wrote an article for The Village  


#### 4. Loading the Parsed Corpus

In [9]:
df_ocr = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/01_parsed_corpus.csv")
print(df_ocr.shape)
print(df_ocr.head())

(26941, 4)
                                         source_file  page_number  \
0  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
1  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
2  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
3  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
4  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   

            region_type                                               text  
0      issue_front_page                                              THIRD  
1  collective_strucutre  Editorial Collective: Lula Mae Blocton, Yvonne...  
2             paragraph  Heresies is an idea-oriented journal devoted t...  
3             paragraph  Heresies is structured as a collective of femi...  
4             paragraph  As women, we are aware that historically the c...  


#### 5. Calculating CER

To do so, I need to compare the Ground Truth to the transcription of the same pages.
I created this with the help of Claude. Claude helped me to compile the transcribed pages which match the gt pages and helped me write the code for the cell calculating the CER.
The prompt for this can be found here: /Users/sophiehamann/master-thesis-code/data/processed/training_data/ocr_pages_for_cer
The script was also made by Claude and can be found here: /Users/sophiehamann/master-thesis-code/src/collect_ocr_pages_for_cer.py

for calculating the CER I used the official [Transkribus documentation formular:](https://blog.transkribus.org/en/how-is-the-cer-calculated-in-transkribus) CER = [ (i + s + d) / n ]*100

In [15]:
from jiwer import cer, process_characters
from lxml import etree
from pathlib import Path

NAMESPACE = {"pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}

GT_PAGE_DIR = Path("/Users/sophiehamann/master-thesis-code/data/processed/training_data/ground_truth/page")
OCR_DIR     = Path("/Users/sophiehamann/master-thesis-code/data/processed/training_data/ocr_pages_for_cer")

def extract_page_text(xml_path):
    """Extract all TextLine Unicode texts from a PAGE XML, joined by space."""
    tree = etree.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall(".//pc:TextLine", NAMESPACE):
        u = tl.find("pc:TextEquiv/pc:Unicode", NAMESPACE)
        if u is not None and u.text and u.text.strip():
            lines.append(u.text.strip())
    return " ".join(lines)

# Build OCR lookup: gt_prefix (e.g. "0001_p002") -> ocr_file path
ocr_lookup = {}
for ocr_file in sorted(OCR_DIR.glob("*.xml")):
    prefix = ocr_file.stem.split("__")[0]
    ocr_lookup[prefix] = ocr_file

# Collect matched GT and OCR texts in the same page order
gt_texts  = []
ocr_texts = []
missing   = []

for gt_file in sorted(GT_PAGE_DIR.glob("*.xml")):
    prefix = gt_file.stem
    if prefix in ocr_lookup:
        gt_texts.append(extract_page_text(gt_file))
        ocr_texts.append(extract_page_text(ocr_lookup[prefix]))
    else:
        missing.append(prefix)

print(f"Pages compared: {len(gt_texts)}")
if missing:
    print(f"GT pages with no OCR match: {missing}")

gt_text  = " ".join(gt_texts)
ocr_text = " ".join(ocr_texts)

# Overall CER
error_rate = cer(gt_text, ocr_text)
print(f"\nCharacter Error Rate: {error_rate:.2%}")

# Breakdown by error type
details = process_characters(gt_text, ocr_text)
print(f"Insertions:    {details.insertions}")
print(f"Substitutions: {details.substitutions}")
print(f"Deletions:     {details.deletions}")
print(f"Total characters in ground truth: {details.hits + details.deletions + details.substitutions}")


Pages compared: 135

Character Error Rate: 8.68%
Insertions:    12419
Substitutions: 7453
Deletions:     27533
Total characters in ground truth: 546419


#### 6. Saving the Results

In [18]:
results = pd.DataFrame([{
    "total_gt_regions":  len(df_gt),
    "total_ocr_regions": len(df_ocr),
    "CER":               error_rate
}])

results.to_csv("/Users/sophiehamann/master-thesis-code/data/processed/02_cer_results.csv", index=False)
print(results)

   total_gt_regions  total_ocr_regions       CER
0              1490              26941  0.086756


#### 7. Quick interpretation: 

I have a CER of 8.69%. Hence, it is below 10%. Official Transkribus documentation considers this "accurate enough to produce "useful" transcriptions, even if they still contain a few errors." ([Source](https://blog.transkribus.org/en/how-is-the-cer-calculated-in-transkribus)).
This means, that the Layout detection, Baseline detection and Text Recognition Model as well as the manual corrections worked well for further analyses with this dataset/corpus.